# Task 1: Prepare and Scale the Data

In [10]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Load the preprocessed dataset.
df = pd.read_csv("Telco_Customer_Churn_Preprocessed")

# Split features and target.
X = df.drop("Churn", axis=1)

# Store the target column.
y = df["Churn"]

# Split the dataset.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Create the scaler.
scaler = StandardScaler()

# Fit and transform training data.
X_train_s = scaler.fit_transform(X_train)

# Transform test data.
X_test_s = scaler.transform(X_test)

# Display dataset shapes.
print(X_train_s.shape)
print(X_test_s.shape)

(5634, 13)
(1409, 13)


# Task 2 — Train an SVM Classifier

In [11]:
# Start the timer.
start = time.time()

# Create the SVM classifier.
svm = SVC(kernel="rbf", random_state=42)

# Train the model.
svm.fit(X_train_s, y_train)

# Calculate training time.
svm_time = time.time() - start

# Predict the test labels.
y_pred_svm = svm.predict(X_test_s)

# Print the accuracy score.
print(f"SVM Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}")

# Print the F1 score.
print(f"SVM F1: {f1_score(y_test, y_pred_svm):.4f}")

# Print the training time.
print(f"SVM Training Time: {svm_time:.2f}s")

# Print the classification report.
print(classification_report(y_test, y_pred_svm))

SVM Accuracy: 0.7835
SVM F1: 0.5468
SVM Training Time: 1.44s
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.62      0.49      0.55       374

    accuracy                           0.78      1409
   macro avg       0.72      0.69      0.70      1409
weighted avg       0.77      0.78      0.78      1409



# Task 3 — Train KNN with K=5

In [12]:
# Start the timer.
start = time.time()

# Create the KNN classifier.
knn5 = KNeighborsClassifier(n_neighbors=5)

# Train the model.
knn5.fit(X_train_s, y_train)

# Calculate training time.
knn5_time = time.time() - start

# Predict the test labels.
y_pred_knn5 = knn5.predict(X_test_s)

# Print the accuracy score.
print(f"KNN (K=5) Accuracy: {accuracy_score(y_test, y_pred_knn5):.4f}")

# Print the F1 score.
print(f"KNN (K=5) F1: {f1_score(y_test, y_pred_knn5):.4f}")

# Print the training time.
print(f"KNN (K=5) Training Time: {knn5_time:.2f}s")

# Print the classification report.
print(classification_report(y_test, y_pred_knn5))

KNN (K=5) Accuracy: 0.7807
KNN (K=5) F1: 0.5476
KNN (K=5) Training Time: 0.01s
              precision    recall  f1-score   support

           0       0.83      0.88      0.86      1035
           1       0.61      0.50      0.55       374

    accuracy                           0.78      1409
   macro avg       0.72      0.69      0.70      1409
weighted avg       0.77      0.78      0.77      1409



# Task 4 — Experiment with Different K Values

In [13]:
# Create a list of K values.
k_values = [3, 5, 10]

# Create an empty list for the results.
results = []

# Train and evaluate each KNN model.
for k in k_values:

    # Create the classifier.
    knn = KNeighborsClassifier(n_neighbors=k)

    # Train the classifier.
    knn.fit(X_train_s, y_train)

    # Predict the test labels.
    predictions = knn.predict(X_test_s)

    # Calculate accuracy.
    accuracy = accuracy_score(y_test, predictions)

    # Calculate the F1 score.
    f1 = f1_score(y_test, predictions)

    # Save the results.
    results.append([k, accuracy, f1])

# Create the comparison table.
comparison = pd.DataFrame(
    results,
    columns=["K Value", "Accuracy", "F1"]
)

# Display the comparison table.
print(comparison)

# Find the best K value.
best_k = comparison.loc[comparison["F1"].idxmax()]

# Print the best K value.
print("\nBest K:")
print(best_k)

   K Value  Accuracy        F1
0        3  0.755145  0.506438
1        5  0.780696  0.547584
2       10  0.784954  0.502463

Best K:
K Value     5.000000
Accuracy    0.780696
F1          0.547584
Name: 1, dtype: float64


# Task 5 — SVM vs Best KNN

In [14]:
# Get the best K value.
best_k_value = int(best_k["K Value"])

# Start the timer.
start = time.time()

# Create the best KNN classifier.
best_knn = KNeighborsClassifier(n_neighbors=best_k_value)

# Train the classifier.
best_knn.fit(X_train_s, y_train)

# Calculate training time.
best_knn_time = time.time() - start

# Predict the test labels.
best_predictions = best_knn.predict(X_test_s)

# Calculate accuracy.
best_accuracy = accuracy_score(y_test, best_predictions)

# Calculate the F1 score.
best_f1 = f1_score(y_test, best_predictions)

# Create the summary table.
summary = pd.DataFrame({
    "Model": ["SVM (RBF)", f"KNN (K={best_k_value})"],
    "Accuracy": [accuracy_score(y_test, y_pred_svm), best_accuracy],
    "F1": [f1_score(y_test, y_pred_svm), best_f1],
    "Training Time": [svm_time, best_knn_time]
})

# Display the summary table.
print(summary)

       Model  Accuracy        F1  Training Time
0  SVM (RBF)  0.783534  0.546805       1.439336
1  KNN (K=5)  0.780696  0.547584       0.012056
